# Reservoir Water Level Forecasting Pipeline

This notebook implements an end-to-end pipeline for predicting reservoir water levels from satellite imagery using:
- **YOLOv11**: Semantic segmentation of water bodies from satellite images
- **Encoder-Decoder ConvLSTM**: Spatiotemporal forecasting of reservoir mask evolution (adapted from jhhuang96/ConvLSTM-PyTorch)
- **Pearson Interpolation**: Correlation-based interpolation for handling sparse temporal observations

The ConvLSTM uses GroupNorm for stable training on binary masks.

**Loss function: MSE (as per Section 2.2.3 of the paper)**

In [1]:
!pip install gdown ultralytics -q
!gdown --id 1zoTQfBNpLcxM3G-uCKUUGjyEvih_hASA
!unrar x -o+ ankhe_dataset.rar ankhe_dataset/

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1zoTQfBNpLcxM3G-uCKUUGjyEvih_hASA
From (redirected): https://drive.google.com/uc?id=1zoTQfBNpLcxM3G-uCKUUGjyEvih_hASA&confirm=t&uuid=57053f6e-7db5-4aa5-ad06-c98926581074
To: /content/ankhe_dataset.rar
100% 62.9M/62.9M [00:00<00:00, 86.2MB/s]

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from ankhe_dataset.rar

Extracting  ankhe_dataset/masks/img_2017-01-28_03-24-03.png                0%  OK 
Extracting  ankhe_dataset/masks/img_2017-02-07_03-24-04.png                0%  OK 
Extracting  ankhe_dataset/masks/img_2017-04-08_03-24-08.png                0%  OK 
Extracting  ankhe_dataset/masks/img_2017-09-05_03-24-09.png                0% 

In [2]:
import gc
import random
import re
import shutil
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import yaml
from skimage.util import img_as_float32
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from ultralytics import YOLO

In [3]:
@dataclass
class PipelineConfig:
    """Configuration for the water level forecasting pipeline."""

    base_dir: Path = field(default_factory=Path.cwd)
    dataset_name: str = "ankhe_dataset"
    image_size: Tuple[int, int] = (320, 320)

    # Data splits (chronological: train first, then val, then test)
    train_ratio: float = 0.6
    val_ratio: float = 0.15
    test_ratio: float = 0.25

    # ConvLSTM parameters
    convlstm_epochs: int = 40
    convlstm_batch_size: int = 1
    convlstm_learning_rate: float = 5e-2
    convlstm_sequence_length: int = 5
    convlstm_accumulation_steps: int = 2

    # YOLO parameters
    yolo_epochs: int = 100
    yolo_image_size: int = 320

    # Water level calculation
    pixel_area_km2: float = 0.0001
    seed: int = 42

    def __post_init__(self):
        self.raw_image_dir = self.base_dir / self.dataset_name / "images"
        self.raw_mask_dir = self.base_dir / self.dataset_name / "masks"
        self.processed_dir = self.base_dir / "processed"
        self.yolo_image_dir = self.processed_dir / "images"
        self.yolo_label_dir = self.processed_dir / "labels"
        self.convlstm_mask_dir = self.processed_dir / "masks"
        self.dense_mask_dir = self.processed_dir / "dense_masks"
        self.split_dir = self.processed_dir / "splits"

    def create_directories(self):
        for d in [self.processed_dir, self.yolo_image_dir, self.yolo_label_dir,
                  self.convlstm_mask_dir, self.dense_mask_dir, self.split_dir]:
            d.mkdir(parents=True, exist_ok=True)

    def set_random_seeds(self):
        random.seed(self.seed)
        np.random.seed(self.seed)
        torch.manual_seed(self.seed)


cfg = PipelineConfig()
cfg.create_directories()
cfg.set_random_seeds()

print(f"Dataset: {cfg.raw_image_dir}")

Dataset: /content/ankhe_dataset/images


In [4]:
def apply_mmse_filter(image: np.ndarray, kernel_size: int = 3) -> np.ndarray:
    """Apply MMSE denoising using local Wiener filtering."""
    channels = cv2.split(image) if image.ndim == 3 else [image]
    filtered = []

    for ch in channels:
        ch_float = img_as_float32(ch)
        squared_blur = cv2.GaussianBlur(ch_float ** 2, (kernel_size, kernel_size), 0)
        blur_squared = cv2.GaussianBlur(ch_float, (kernel_size, kernel_size), 0) ** 2
        noise_var = np.mean(squared_blur - blur_squared)

        local_mean = cv2.GaussianBlur(ch_float, (kernel_size, kernel_size), 0)
        local_var = np.maximum(squared_blur - blur_squared, 1e-6)
        signal_var = np.maximum(local_var - noise_var, 0)

        wiener = signal_var / local_var
        result = local_mean + wiener * (ch_float - local_mean)
        filtered.append(np.clip(result, 0, 1))

    return cv2.merge(filtered) if len(filtered) > 1 else filtered[0]

In [5]:
def preprocess_satellite_imagery(config: PipelineConfig) -> List[Dict[str, Path]]:
    """Preprocess satellite images: denoise, resize, and binarize masks."""
    records = []

    for img_path in sorted(config.raw_image_dir.glob("*.png")):
        mask_path = config.raw_mask_dir / img_path.name
        if not mask_path.exists():
            continue

        # Process image
        image = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        image = apply_mmse_filter(img_as_float32(image))
        image = cv2.resize(image, config.image_size, interpolation=cv2.INTER_AREA)

        out_img = config.yolo_image_dir / img_path.name
        cv2.imwrite(str(out_img), (np.clip(image, 0, 1) * 255).astype(np.uint8))

        # Process mask
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, config.image_size, interpolation=cv2.INTER_NEAREST)
        _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

        out_mask = config.convlstm_mask_dir / mask_path.name
        cv2.imwrite(str(out_mask), mask)

        records.append({"img": out_img, "mask": out_mask})

    return records


records = preprocess_satellite_imagery(cfg)
print(f"Processed {len(records)} image-mask pairs")

Processed 71 image-mask pairs


In [6]:
def convert_mask_to_yolo_polygon(mask_path: Path, image_size: Tuple[int, int] = (320, 320),
                                  max_points: int = 200) -> Optional[List[float]]:
    """Convert binary mask to normalized YOLO polygon coordinates."""
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    if not contours:
        return None

    contour = max(contours, key=cv2.contourArea)
    if cv2.contourArea(contour) < 100:
        return None

    # Simplify polygon
    epsilon = 0.001 * cv2.arcLength(contour, True)
    simplified = cv2.approxPolyDP(contour, epsilon, True)

    # Reduce points if needed
    if len(simplified) > max_points:
        indices = np.linspace(0, len(simplified) - 1, max_points, dtype=int)
        simplified = simplified[indices]

    if len(simplified) < 3:
        return None

    # Normalize coordinates
    coords = []
    for pt in simplified.flatten().reshape(-1, 2):
        coords.extend([np.clip(pt[0] / image_size[0], 0, 1),
                       np.clip(pt[1] / image_size[1], 0, 1)])
    return coords


def export_yolo_labels(records: List[Dict[str, Path]], output_dir: Path) -> int:
    """Export YOLO segmentation labels from masks."""
    output_dir.mkdir(parents=True, exist_ok=True)
    count = 0

    for record in records:
        coords = convert_mask_to_yolo_polygon(record["mask"])
        if coords and len(coords) >= 6:
            label_path = output_dir / (Path(record["img"]).stem + ".txt")
            with open(label_path, "w") as f:
                f.write(f"0 {' '.join(f'{c:.6f}' for c in coords)}")
            count += 1

    print(f"Generated {count} YOLO labels")
    return count


export_yolo_labels(records, cfg.yolo_label_dir)

Generated 71 YOLO labels


71

In [7]:
def parse_timestamp(file_path: Path) -> Optional[pd.Timestamp]:
    """Extract YYYY-MM timestamp from filename."""
    match = re.search(r"(20\d{2})[-_]?([01]?\d)", file_path.stem)
    if match:
        year, month = int(match.group(1)), int(match.group(2))
        if 1 <= month <= 12:
            return pd.Timestamp(year=year, month=month, day=1)
    return None


def sort_records_by_timestamp(records: List[Dict[str, Path]]) -> List[Dict[str, Path]]:
    """Sort records chronologically."""
    return sorted(records, key=lambda r: (parse_timestamp(Path(r["mask"])) or pd.Timestamp.max, r["mask"].name))

In [8]:
def write_split_file(paths: List[Path], output: Path):
    with open(output, "w") as f:
        f.writelines(f"{p}\n" for p in paths)


def load_split_file(path: Path) -> List[Path]:
    if not path.exists():
        return []
    with open(path) as f:
        return [Path(line.strip()) for line in f if line.strip()]


def create_dataset_splits(records: List[Dict[str, Path]], config: PipelineConfig,
                          dense_masks: Optional[List[Path]] = None,
                          interpolated_masks: Optional[set] = None) -> Dict[str, List[Path]]:
    sorted_records = sort_records_by_timestamp(records)
    images = [config.yolo_image_dir / Path(r["img"]).name for r in sorted_records]

    train, temp = train_test_split(images, train_size=config.train_ratio, shuffle=False)
    val, test = train_test_split(temp, test_size=config.test_ratio / (config.val_ratio + config.test_ratio), shuffle=False)

    splits = {"train": train, "val": val, "test": test}
    write_split_file(train, config.split_dir / "train.txt")
    write_split_file(val, config.split_dir / "val.txt")
    write_split_file(test, config.split_dir / "test.txt")

    if dense_masks:
        sorted_masks = sorted(dense_masks, key=lambda p: (parse_timestamp(p) or pd.Timestamp.max, p.name))
        interpolated_set = interpolated_masks or set()

        real_masks = [m for m in sorted_masks if m not in interpolated_set]

        real_train, real_temp = train_test_split(real_masks, train_size=config.train_ratio, shuffle=False)
        real_val, real_test = train_test_split(real_temp,
                                                test_size=config.test_ratio / (config.val_ratio + config.test_ratio),
                                                shuffle=False)

        train_timestamps = {parse_timestamp(p) for p in real_train if parse_timestamp(p) is not None}
        if train_timestamps:
            train_end_ts = max(train_timestamps)
        else:
            train_end_ts = pd.Timestamp.min

        train_interp = [m for m in sorted_masks
                        if m in interpolated_set and
                        (parse_timestamp(m) is not None and parse_timestamp(m) <= train_end_ts)]

        train_m = sorted(real_train + train_interp,
                         key=lambda p: (parse_timestamp(p) or pd.Timestamp.max, p.name))

        val_m = real_val
        test_m = real_test

        splits.update({"train_masks": train_m, "val_masks": val_m, "test_masks": test_m})
        write_split_file(train_m, config.split_dir / "train_masks.txt")
        write_split_file(val_m, config.split_dir / "val_masks.txt")
        write_split_file(test_m, config.split_dir / "test_masks.txt")

        n_train_real = len(real_train)
        n_train_interp = len(train_interp)
        print(f"\n=== DATA SPLIT SUMMARY (No Leakage) ===")
        print(f"Training masks: {len(train_m)} total ({n_train_real} real + {n_train_interp} interpolated)")
        print(f"Validation masks: {len(val_m)} (ALL REAL - no interpolated)")
        print(f"Test masks: {len(test_m)} (ALL REAL - no interpolated)")
        print(f"========================================\n")

    print(f"Splits: {{{', '.join(f'{k}: {len(v)}' for k, v in splits.items())}}}")
    return splits


splits = None

In [9]:
def augment_image_mask(image: np.ndarray, mask: np.ndarray) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Generate augmented pairs via flips and rotations."""
    return [
        (cv2.flip(image, 1), cv2.flip(mask, 1)),
        (cv2.flip(image, 0), cv2.flip(mask, 0)),
        (cv2.flip(image, -1), cv2.flip(mask, -1)),
        (cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE), cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE)),
        (cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE), cv2.rotate(mask, cv2.ROTATE_90_COUNTERCLOCKWISE)),
    ]


def augment_training_data(train_records: List[Dict[str, Path]], all_records: List[Dict[str, Path]],
                          config: PipelineConfig) -> List[Dict[str, Path]]:
    """Apply augmentation to training data only."""
    augmented = []

    for record in tqdm(train_records, desc="Augmenting"):
        image = cv2.cvtColor(cv2.imread(str(record["img"])), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(record["mask"]), cv2.IMREAD_GRAYSCALE)
        stem = Path(record["img"]).stem

        for i, (aug_img, aug_mask) in enumerate(augment_image_mask(image, mask)):
            out_img = config.yolo_image_dir / f"{stem}_aug{i}.png"
            out_mask = config.convlstm_mask_dir / f"{stem}_aug{i}.png"
            cv2.imwrite(str(out_img), cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))
            cv2.imwrite(str(out_mask), aug_mask)
            augmented.append({"img": out_img, "mask": out_mask, "is_augmented": True})

    print(f"Generated {len(augmented)} augmented samples")
    return all_records + augmented

In [10]:
import torch
import torch.nn as nn
import numpy as np
import cv2
import gc
import shutil
import yaml
from pathlib import Path
from typing import List, Optional, Tuple, Dict
from ultralytics import YOLO

# Clear cache before defining models
torch.cuda.empty_cache()
gc.collect()

loss_fn = nn.MSELoss()

def pearson_interpolation(m1: np.ndarray, m2: np.ndarray, alpha: float) -> np.ndarray:
    """
    Pearson-weighted Interpolation:
    Kết hợp giữa Time-weighted (alpha) và Pearson Correlation (r).
    """
    # 1. Tính toán r (như cũ)
    f1 = m1.astype(np.float32).flatten()
    f2 = m2.astype(np.float32).flatten()
    covariance = np.cov(f1, f2)[0, 1]
    sigma1 = np.std(f1)
    sigma2 = np.std(f2)

    r = 0.0
    if sigma1 > 1e-8 and sigma2 > 1e-8:
        r = covariance / (sigma1 * sigma2)

    # Clip r trong khoảng [0, 1] để làm trọng số
    r = np.clip(r, 0, 1)


    m1_float = m1.astype(np.float32) / 255.0
    m2_float = m2.astype(np.float32) / 255.0


    if r > 0.5
        interpolated = (1 - alpha) * m1_float + alpha * m2_float
    else:
        interpolated = m1_float if alpha < 0.5 else m2_float

    _, binary = cv2.threshold((interpolated * 255).astype(np.uint8), 127, 255, cv2.THRESH_BINARY)
    return binary


def interpolate_causal(seq: List[Optional[np.ndarray]], image_size: Tuple[int, int]) -> List[np.ndarray]:
    result, last = [], None
    for m in seq:
        if m is not None:
            result.append(m)
            last = m
        elif last is not None:
            result.append(last.copy())
        else:
            result.append(np.zeros(image_size, dtype=np.uint8))
    return result


def interpolate_bidirectional(seq: List[Optional[np.ndarray]], image_size: Tuple[int, int]) -> List[np.ndarray]:
    result = []
    for i, m in enumerate(seq):
        if m is not None:
            result.append(m)
            continue
        prev_idx, next_idx = None, None
        for j in range(i - 1, -1, -1):
            if seq[j] is not None:
                prev_idx = j
                break
        for j in range(i + 1, len(seq)):
            if seq[j] is not None:
                next_idx = j
                break
        if prev_idx is None and next_idx is None:
            result.append(np.zeros(image_size, dtype=np.uint8))
        elif prev_idx is None:
            result.append(seq[next_idx].copy())
        elif next_idx is None:
            result.append(seq[prev_idx].copy())
        else:
            alpha = (i - prev_idx) / (next_idx - prev_idx)
            result.append(pearson_interpolation(seq[prev_idx], seq[next_idx], alpha))
    return result


class MIMN(nn.Module):
    def __init__(self, num_hidden, height, width, filter_size=3):
        super(MIMN, self).__init__()
        self.num_hidden = num_hidden
        self.padding = (filter_size - 1) // 2

        num_groups = max(1, (4 * num_hidden) // 32)

        self.conv_h = nn.Sequential(
            nn.Conv2d(num_hidden, 4 * num_hidden, filter_size, 1, self.padding),
            nn.GroupNorm(num_groups, 4 * num_hidden)
        )
        self.conv_x = nn.Sequential(
            nn.Conv2d(num_hidden, 4 * num_hidden, filter_size, 1, self.padding),
            nn.GroupNorm(num_groups, 4 * num_hidden)
        )

        self.ct_weight = nn.Parameter(torch.randn(2 * num_hidden, height, width) * 0.1)
        self.oc_weight = nn.Parameter(torch.randn(num_hidden, height, width) * 0.1)

    def forward(self, x, h_t, c_t):
        h_concat = self.conv_h(h_t)
        x_concat = self.conv_x(x)

        i_h, g_h, f_h, o_h = torch.split(h_concat, self.num_hidden, dim=1)
        i_x, g_x, f_x, o_x = torch.split(x_concat, self.num_hidden, dim=1)

        ct_activation = c_t.repeat(1, 2, 1, 1) * self.ct_weight
        i_c, f_c = torch.split(ct_activation, self.num_hidden, dim=1)

        i_ = i_x + i_h + i_c
        f_ = f_x + f_h + f_c
        g_ = g_x + g_h
        o_ = o_x + o_h

        i_ = torch.sigmoid(i_)
        f_ = torch.sigmoid(f_ + 1.0)
        g_ = torch.tanh(g_)

        c_new = f_ * c_t + i_ * g_

        o_c = c_new * self.oc_weight
        o_ = torch.sigmoid(o_ + o_c)

        h_new = o_ * torch.tanh(c_new)

        return h_new, c_new


class MIMBlock(nn.Module):
    def __init__(self, num_hidden, height, width, filter_size=3):
        super(MIMBlock, self).__init__()
        self.num_hidden = num_hidden
        self.padding = (filter_size - 1) // 2
        num_groups = max(1, (4 * num_hidden) // 32)
        num_groups_3 = max(1, (3 * num_hidden) // 32)

        self.mims_h = nn.Sequential(
            nn.Conv2d(num_hidden, 4 * num_hidden, filter_size, 1, self.padding),
            nn.GroupNorm(num_groups, 4 * num_hidden)
        )
        self.mims_x = nn.Sequential(
            nn.Conv2d(num_hidden, 4 * num_hidden, filter_size, 1, self.padding),
            nn.GroupNorm(num_groups, 4 * num_hidden)
        )
        self.mims_ct_weight = nn.Parameter(torch.randn(2 * num_hidden, height, width) * 0.1)
        self.mims_oc_weight = nn.Parameter(torch.randn(num_hidden, height, width) * 0.1)

        self.t_cc = nn.Sequential(
            nn.Conv2d(num_hidden, 3 * num_hidden, filter_size, 1, self.padding),
            nn.GroupNorm(num_groups_3, 3 * num_hidden)
        )
        self.s_cc = nn.Sequential(
            nn.Conv2d(num_hidden, 4 * num_hidden, filter_size, 1, self.padding),
            nn.GroupNorm(num_groups, 4 * num_hidden)
        )
        self.x_cc = nn.Sequential(
            nn.Conv2d(num_hidden, 4 * num_hidden, filter_size, 1, self.padding),
            nn.GroupNorm(num_groups, 4 * num_hidden)
        )

        self.c_reduce = nn.Conv2d(2 * num_hidden, num_hidden, 1, 1, 0)

    def MIMS(self, x, h_t, c_t):
        h_concat = self.mims_h(h_t)
        i_h, g_h, f_h, o_h = torch.split(h_concat, self.num_hidden, dim=1)

        ct_activation = c_t.repeat(1, 2, 1, 1) * self.mims_ct_weight
        i_c, f_c = torch.split(ct_activation, self.num_hidden, dim=1)

        i_ = i_h + i_c
        f_ = f_h + f_c
        g_ = g_h
        o_ = o_h

        if x is not None:
            x_concat = self.mims_x(x)
            i_x, g_x, f_x, o_x = torch.split(x_concat, self.num_hidden, dim=1)
            i_ = i_ + i_x
            f_ = f_ + f_x
            g_ = g_ + g_x
            o_ = o_ + o_x

        i_ = torch.sigmoid(i_)
        f_ = torch.sigmoid(f_ + 1.0)
        g_ = torch.tanh(g_)

        c_new = f_ * c_t + i_ * g_

        o_c = c_new * self.mims_oc_weight
        h_new = torch.sigmoid(o_ + o_c) * torch.tanh(c_new)

        return h_new, c_new

    def forward(self, x, diff_h, h, c, m, convlstm_c):
        t_cc = self.t_cc(h)
        s_cc = self.s_cc(m)
        x_cc = self.x_cc(x)

        i_s, g_s, f_s, o_s = torch.split(s_cc, self.num_hidden, dim=1)
        i_t, g_t, o_t = torch.split(t_cc, self.num_hidden, dim=1)
        i_x, g_x, f_x, o_x = torch.split(x_cc, self.num_hidden, dim=1)

        i = torch.sigmoid(i_x + i_t)
        i_ = torch.sigmoid(i_x + i_s)
        g = torch.tanh(g_x + g_t)
        g_ = torch.tanh(g_x + g_s)
        f_ = torch.sigmoid(f_x + f_s + 1.0)
        o = torch.sigmoid(o_x + o_t + o_s)

        new_m = f_ * m + i_ * g_

        c, new_convlstm_c = self.MIMS(diff_h, c, convlstm_c)
        new_c = c + i * g

        cell = torch.cat([new_c, new_m], dim=1)
        cell = self.c_reduce(cell)
        new_h = o * torch.tanh(cell)

        return new_h, new_c, new_m, new_convlstm_c


class SpatioTemporalLSTMCell(nn.Module):
    def __init__(self, input_channels, num_hidden, filter_size=3):
        super(SpatioTemporalLSTMCell, self).__init__()
        self.num_hidden = num_hidden
        self.padding = (filter_size - 1) // 2
        num_groups = max(1, (4 * num_hidden) // 32)

        self.conv_t = nn.Sequential(
            nn.Conv2d(num_hidden, 4 * num_hidden, filter_size, 1, self.padding),
            nn.GroupNorm(num_groups, 4 * num_hidden)
        )
        self.conv_s = nn.Sequential(
            nn.Conv2d(num_hidden, 4 * num_hidden, filter_size, 1, self.padding),
            nn.GroupNorm(num_groups, 4 * num_hidden)
        )
        self.conv_x = nn.Sequential(
            nn.Conv2d(input_channels, 4 * num_hidden, filter_size, 1, self.padding),
            nn.GroupNorm(num_groups, 4 * num_hidden)
        )
        self.conv_last = nn.Conv2d(2 * num_hidden, num_hidden, 1, 1, 0)

    def forward(self, x, h, c, m):
        t_cc = self.conv_t(h)
        s_cc = self.conv_s(m)
        x_cc = self.conv_x(x)

        i_s, g_s, f_s, o_s = torch.split(s_cc, self.num_hidden, dim=1)
        i_t, g_t, f_t, o_t = torch.split(t_cc, self.num_hidden, dim=1)
        i_x, g_x, f_x, o_x = torch.split(x_cc, self.num_hidden, dim=1)

        i = torch.sigmoid(i_x + i_t)
        i_ = torch.sigmoid(i_x + i_s)
        g = torch.tanh(g_x + g_t)
        g_ = torch.tanh(g_x + g_s)
        f = torch.sigmoid(f_x + f_t + 1.0)
        f_ = torch.sigmoid(f_x + f_s + 1.0)
        o = torch.sigmoid(o_x + o_t + o_s)

        new_m = f_ * m + i_ * g_
        new_c = f * c + i * g

        cell = torch.cat([new_c, new_m], dim=1)
        cell = self.conv_last(cell)
        new_h = o * torch.tanh(cell)

        return new_h, new_c, new_m


class ReservoirConvLSTM(nn.Module):
    def __init__(self, input_dim=1, num_hidden=16, num_layers=3, shape=(320, 320)):
        super().__init__()
        self.num_layers = num_layers
        self.num_hidden = num_hidden
        self.shape = shape

        self.stlstm_layer = SpatioTemporalLSTMCell(input_dim, num_hidden)

        self.mim_layers = nn.ModuleList()
        self.mimn_layers = nn.ModuleList()

        for i in range(1, num_layers):
            self.mim_layers.append(
                MIMBlock(num_hidden, shape[0], shape[1])
            )
            self.mimn_layers.append(
                MIMN(num_hidden, shape[0], shape[1])
            )

        self.conv_last = nn.Conv2d(num_hidden, input_dim, 1, 1, 0)

    def forward(self, seq):
        device = seq.device
        B, T, C, H, W = seq.shape

        h_t = [torch.zeros(B, self.num_hidden, H, W).to(device) for _ in range(self.num_layers)]
        c_t = [torch.zeros(B, self.num_hidden, H, W).to(device) for _ in range(self.num_layers)]
        convlstm_c_t = [torch.zeros(B, self.num_hidden, H, W).to(device) for _ in range(self.num_layers)]

        diff_h_t = [torch.zeros(B, self.num_hidden, H, W).to(device) for _ in range(self.num_layers - 1)]
        diff_c_t = [torch.zeros(B, self.num_hidden, H, W).to(device) for _ in range(self.num_layers - 1)]

        st_memory = torch.zeros(B, self.num_hidden, H, W).to(device)

        prev_h0 = None
        gen_images = []

        for t in range(T):
            x = seq[:, t, ...]

            if t == 0:
                prev_h0 = torch.zeros_like(h_t[0])

            h_t[0], c_t[0], st_memory = self.stlstm_layer(x, h_t[0], c_t[0], st_memory)

            for i in range(1, self.num_layers):
                if t > 0:
                    if i == 1:
                        diff_input = h_t[0] - prev_h0
                    else:
                        diff_input = diff_h_t[i-2]

                    diff_h_t[i-1], diff_c_t[i-1] = self.mimn_layers[i-1](diff_input, diff_h_t[i-1], diff_c_t[i-1])
                else:
                    diff_input = torch.zeros(B, self.num_hidden, H, W).to(device)
                    diff_h_t[i-1], diff_c_t[i-1] = self.mimn_layers[i-1](diff_input, diff_h_t[i-1], diff_c_t[i-1])

                h_t[i], c_t[i], st_memory, convlstm_c_t[i] = self.mim_layers[i-1](
                    h_t[i-1], diff_h_t[i-1], h_t[i], c_t[i], st_memory, convlstm_c_t[i]
                )

            prev_h0 = h_t[0].clone()

            x_gen = self.conv_last(h_t[-1])
            gen_images.append(x_gen)

        return torch.sigmoid(gen_images[-1])


def train_yolo_segmentation(config: PipelineConfig) -> YOLO:
    yolo_data_dir = config.processed_dir / "yolo_data"
    for split in ["train", "val"]:
        (yolo_data_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (yolo_data_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

    train_imgs = load_split_file(config.split_dir / "train.txt")
    val_imgs = load_split_file(config.split_dir / "val.txt")

    for img_path in train_imgs:
        if img_path.exists():
            shutil.copy(str(img_path), str(yolo_data_dir / "images" / "train" / img_path.name))
            label_path = config.yolo_label_dir / (img_path.stem + ".txt")
            if label_path.exists():
                shutil.copy(str(label_path), str(yolo_data_dir / "labels" / "train" / label_path.name))

    for img_path in val_imgs:
        if img_path.exists():
            shutil.copy(str(img_path), str(yolo_data_dir / "images" / "val" / img_path.name))
            label_path = config.yolo_label_dir / (img_path.stem + ".txt")
            if label_path.exists():
                shutil.copy(str(label_path), str(yolo_data_dir / "labels" / "val" / label_path.name))

    data_yaml = {"path": str(yolo_data_dir.absolute()), "train": "images/train", "val": "images/val", "names": {0: "water"}}
    yaml_path = yolo_data_dir / "data.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(data_yaml, f)

    print(f"YOLO dataset: {len(train_imgs)} train, {len(val_imgs)} val images")
    model = YOLO("yolo11x-seg.pt") # Đã sửa thành YOLOv11x-seg
    model.train(data=str(yaml_path), epochs=config.yolo_epochs, imgsz=config.yolo_image_size, batch=8, patience=20,
                save=True, project=str(config.processed_dir / "yolo_runs"), name="water_seg", exist_ok=True, verbose=True)
    best_path = config.processed_dir / "yolo_runs" / "water_seg" / "weights" / "best.pt"
    if best_path.exists():
        model = YOLO(str(best_path))
    return model


def yolo_predict_mask(model: YOLO, image_path: Path, config: PipelineConfig) -> Optional[np.ndarray]:
    results = model.predict(str(image_path), imgsz=config.yolo_image_size, verbose=False)
    if results and results[0].masks is not None:
        mask = results[0].masks.data[0].cpu().numpy()
        mask = cv2.resize(mask, config.image_size, interpolation=cv2.INTER_NEAREST)
        return (mask > 0.5).astype(np.uint8) * 255
    return None


def evaluate_yolo_on_test(model: YOLO, config: PipelineConfig) -> Dict[str, float]:
    test_imgs = load_split_file(config.split_dir / "test.txt")
    ious, f1s = [], []
    for img_path in test_imgs:
        if not img_path.exists():
            continue
        gt_mask_path = config.convlstm_mask_dir / img_path.name
        if not gt_mask_path.exists():
            continue
        gt_mask = cv2.imread(str(gt_mask_path), cv2.IMREAD_GRAYSCALE)
        pred_mask = yolo_predict_mask(model, img_path, config)
        if pred_mask is not None:
            ious.append(compute_iou(gt_mask, pred_mask))
            f1s.append(compute_f1(gt_mask, pred_mask))
    results = {"mean_iou": np.mean(ious) if ious else 0.0, "mean_f1": np.mean(f1s) if f1s else 0.0, "num_samples": len(ious)}
    print(f"YOLO Test: IoU={results['mean_iou']:.4f}, F1={results['mean_f1']:.4f} ({results['num_samples']} samples)")
    return results


def compute_iou(mask1: np.ndarray, mask2: np.ndarray) -> float:
    m1 = (mask1 > 127).astype(bool) if mask1.max() > 1 else mask1.astype(bool)
    m2 = (mask2 > 127).astype(bool) if mask2.max() > 1 else mask2.astype(bool)
    inter = np.logical_and(m1, m2).sum()
    union = np.logical_or(m1, m2).sum()
    return inter / (union + 1e-8)


def compute_f1(mask1: np.ndarray, mask2: np.ndarray) -> float:
    m1 = (mask1 > 127).astype(bool) if mask1.max() > 1 else mask1.astype(bool)
    m2 = (mask2 > 127).astype(bool) if mask2.max() > 1 else mask2.astype(bool)
    tp = np.logical_and(m1, m2).sum()
    fp = np.logical_and(m1, ~m2).sum()
    fn = np.logical_and(~m1, m2).sum()
    prec = tp / (tp + fp + 1e-8)
    rec = tp / (tp + fn + 1e-8)
    return 2 * prec * rec / (prec + rec + 1e-8)


print("Models defined: ReservoirConvLSTM, YOLOv11, MSE loss")

Models defined: ReservoirConvLSTM, YOLOv11, MSE loss


In [11]:
class TemporalMaskDataset(Dataset):
    def __init__(self, files: List[Path], seq_len: int = 5):
        self.files = list(files)
        self.seq_len = seq_len

    def __len__(self): return max(1, len(self.files) - self.seq_len)

    def __getitem__(self, idx):
        idx = min(idx, max(0, len(self.files) - self.seq_len - 1))
        masks = [cv2.imread(str(self.files[idx + k]), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
                 for k in range(self.seq_len + 1)]
        return (torch.from_numpy(np.stack([m[None] for m in masks[:-1]])).float(),
                torch.from_numpy(masks[-1][None]).float())


def train_convlstm(config: PipelineConfig) -> nn.Module:
    torch.cuda.empty_cache()
    gc.collect()
    train_files = load_split_file(config.split_dir / "train_masks.txt")
    val_files = load_split_file(config.split_dir / "val_masks.txt")
    if not train_files:
        all_files = sorted(config.dense_mask_dir.glob("*.png"))
        train_files, val_files = train_test_split(all_files, test_size=0.2, shuffle=False) if all_files else ([], [])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ReservoirConvLSTM().to(device)
    loader = DataLoader(TemporalMaskDataset(train_files, config.convlstm_sequence_length), config.convlstm_batch_size, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=config.convlstm_learning_rate, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, config.convlstm_epochs)
    mse = nn.MSELoss()

    best_loss, best_state = float("inf"), None
    print(f"Training ConvLSTM: {len(train_files)} train, {len(val_files)} val masks")

    for epoch in range(config.convlstm_epochs):
        model.train()
        losses = []
        opt.zero_grad()
        for i, (seq, target) in enumerate(loader):
            pred = model(seq.to(device))
            loss = mse(pred, target.to(device)) / config.convlstm_accumulation_steps
            loss.backward()
            if (i + 1) % config.convlstm_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                opt.zero_grad()
            losses.append(loss.item() * config.convlstm_accumulation_steps)
        sched.step()

        model.eval()
        with torch.no_grad():
            val_loader = DataLoader(TemporalMaskDataset(val_files, config.convlstm_sequence_length), config.convlstm_batch_size)
            val_loss = np.mean([mse(model(x.to(device)), y.to(device)).item() for x, y in val_loader])
        if val_loss < best_loss:
            best_loss, best_state = val_loss, {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1}: train={np.mean(losses):.4f}, val={val_loss:.4f}")

    if best_state:
        model.load_state_dict(best_state)
    print(f"Best val loss: {best_loss:.4f}")
    return model


REAL_TIMESTAMPS: set = set()

def build_dense_masks(records: List[Dict[str, Path]], config: PipelineConfig) -> Tuple[List[Path], set]:
    global REAL_TIMESTAMPS
    sorted_recs = sort_records_by_timestamp(records)
    dated = [(parse_timestamp(Path(r["mask"])), r["mask"]) for r in sorted_recs]
    dated = [(ts, p) for ts, p in dated if ts is not None]
    REAL_TIMESTAMPS = {ts for ts, _ in dated}
    if not dated:
        return sorted(config.convlstm_mask_dir.glob("*.png")), set()

    paths_list = [Path(p) for _, p in dated]
    train_paths, _ = train_test_split(paths_list, train_size=config.train_ratio, shuffle=False)
    ts_to_mask = {ts: Path(p) for ts, p in dated}
    train_ts_set = {ts for ts, p in dated if Path(p) in train_paths}

    monthly = pd.date_range(start=min(ts for ts, _ in dated), end=max(ts for ts, _ in dated), freq="MS")
    seq = [cv2.resize(cv2.imread(str(ts_to_mask[ts]), cv2.IMREAD_GRAYSCALE), config.image_size, interpolation=cv2.INTER_NEAREST)
           if ts in ts_to_mask else None for ts in monthly]

    train_end_ts = max(train_ts_set) if train_ts_set else monthly[0]
    train_end = sum(1 for ts in monthly if ts <= train_end_ts)

    print(f"Interpolation: {train_end} train months (bidirectional), {len(monthly) - train_end} val/test months (causal)")

    filled = interpolate_bidirectional(seq[:train_end], config.image_size) + interpolate_causal(seq[train_end:], config.image_size)

    out_paths, interpolated = [], set()
    for ts, m in zip(monthly, filled):
        out = config.dense_mask_dir / f"mask_{ts.strftime('%Y-%m')}.png"
        cv2.imwrite(str(out), m)
        out_paths.append(out)
        if ts not in REAL_TIMESTAMPS:
            interpolated.add(out)
    print(f"Generated {len(out_paths)} dense masks ({len(interpolated)} interpolated)")
    return out_paths, interpolated


DENSE_MASKS, INTERPOLATED_MASKS = build_dense_masks(records, cfg)
splits = create_dataset_splits(records, cfg, DENSE_MASKS, INTERPOLATED_MASKS)

Interpolation: 56 train months (bidirectional), 46 val/test months (causal)
Generated 102 dense masks (52 interpolated)

=== DATA SPLIT SUMMARY (No Leakage) ===
Training masks: 56 total (30 real + 26 interpolated)
Validation masks: 7 (ALL REAL - no interpolated)
Test masks: 13 (ALL REAL - no interpolated)

Splits: {train: 42, val: 10, test: 19, train_masks: 56, val_masks: 7, test_masks: 13}


In [12]:
import numpy as np
import cv2
import shutil
import pandas as pd
from pathlib import Path
from typing import List

# ==========================================
# 1. WATER LEVEL CONVERSION (FROM PAPER TABLE 2)
# ==========================================

# Dữ liệu từ bài báo: Diện tích F (km2)
AREA_POINTS = np.array([
    0.00, 0.56, 0.73, 0.91, 1.12, 1.36, 1.68, 2.01, 2.18,
    2.36, 2.56, 2.78, 3.39, 4.02, 5.00, 6.13, 7.43, 9.00
])

# Dữ liệu từ bài báo: Mực nước Z (m)
WATER_LEVEL_POINTS = np.array([
    413.35, 420.00, 421.00, 422.00, 423.00, 424.00, 425.00, 426.00, 426.50,
    427.00, 427.50, 428.00, 429.00, 430.00, 431.00, 432.00, 433.00, 434.00
])

def convert_area_to_level(area_km2: float) -> float:
    """Nội suy tuyến tính: Diện tích (F) -> Mực nước (Z)."""
    return np.interp(area_km2, AREA_POINTS, WATER_LEVEL_POINTS)

def mask_to_water_level(mask: np.ndarray, pixel_area_km2: float = 0.0001) -> float:
    """
    Chuyển đổi Mask -> Mực nước (Water Level).
    1. Đếm số pixel nước.
    2. Tính diện tích km2 (mặc định Sentinel-2: 100m2 = 0.0001 km2).
    3. Tra bảng để ra mực nước Z.
    """
    if mask is None:
        return 0.0

    # Xử lý mask 0-255 hoặc 0-1
    if np.max(mask) > 1:
        water_pixels = np.sum(mask > 127)
    else:
        water_pixels = np.sum(mask > 0.5)

    area_km2 = water_pixels * pixel_area_km2
    return convert_area_to_level(area_km2)

# ==========================================
# 2. EVALUATION METRICS (RMSE/MAE ON Z)
# ==========================================

def rmse(gt: np.ndarray, pred: np.ndarray) -> float:
    """
    Tính RMSE dựa trên Mực nước (Water Level Z).
    Input có thể là 1 ảnh hoặc 1 batch ảnh.
    """
    # Nếu input là 1 ảnh (H, W), chuyển thành list
    if gt.ndim == 2:
        gt = [gt]
        pred = [pred]

    gt_vals = []
    pred_vals = []

    for i in range(len(gt)):
        z_gt = mask_to_water_level(gt[i])
        z_pred = mask_to_water_level(pred[i])
        gt_vals.append(z_gt)
        pred_vals.append(z_pred)

    gt_arr = np.array(gt_vals)
    pred_arr = np.array(pred_vals)

    return np.sqrt(np.mean((gt_arr - pred_arr) ** 2))

def mae(gt: np.ndarray, pred: np.ndarray) -> float:
    """
    Tính MAE dựa trên Mực nước (Water Level Z).
    """
    if gt.ndim == 2:
        gt = [gt]
        pred = [pred]

    gt_vals = []
    pred_vals = []

    for i in range(len(gt)):
        z_gt = mask_to_water_level(gt[i])
        z_pred = mask_to_water_level(pred[i])
        gt_vals.append(z_gt)
        pred_vals.append(z_pred)

    gt_arr = np.array(gt_vals)
    pred_arr = np.array(pred_vals)

    return np.mean(np.abs(gt_arr - pred_arr))

# ==========================================
# 3. INTERPOLATION PIPELINE
# ==========================================

class InterpolationMethod:
    """Base class for interpolation methods."""
    def __init__(self, name: str):
        self.name = name

    def interpolate(self, m1: np.ndarray, m2: np.ndarray, alpha: float) -> np.ndarray:
        raise NotImplementedError


class PearsonInterp(InterpolationMethod):
    """Pearson correlation-based interpolation (paper method)."""
    def __init__(self):
        super().__init__("pearson")

    def interpolate(self, m1: np.ndarray, m2: np.ndarray, alpha: float) -> np.ndarray:
        # Lưu ý: Đảm bảo hàm pearson_interpolation đã được định nghĩa ở cell trước
        return pearson_interpolation(m1, m2, alpha)


# Available interpolation methods
INTERPOLATION_METHODS = {
    'pearson': PearsonInterp(),
}


def generate_interpolated_masks_safe(train_mask_paths: List[Path], method: InterpolationMethod,
                                     config: PipelineConfig, interp_per_gap: int = 3) -> List[Path]:
    """Generate interpolated masks using ONLY training data (no future leakage)."""
    if len(train_mask_paths) < 2:
        return list(train_mask_paths)

    # Sort theo timestamp hoặc tên
    sorted_paths = sorted(train_mask_paths, key=lambda p: (parse_timestamp(p) or pd.Timestamp.max, p.name))

    output_dir = config.processed_dir / f"interp_{method.name}"
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    all_paths = []
    # Copy file gốc vào thư mục mới
    for i, p in enumerate(sorted_paths):
        out = output_dir / f"orig_{i:03d}_{p.name}"
        shutil.copy(str(p), str(out))
        all_paths.append(out)

    interp_paths = []
    for i in range(len(sorted_paths) - 1):
        m1 = cv2.imread(str(sorted_paths[i]), cv2.IMREAD_GRAYSCALE)
        m2 = cv2.imread(str(sorted_paths[i + 1]), cv2.IMREAD_GRAYSCALE)

        if m1 is None or m2 is None:
            continue

        m1 = cv2.resize(m1, config.image_size, interpolation=cv2.INTER_NEAREST)
        m2 = cv2.resize(m2, config.image_size, interpolation=cv2.INTER_NEAREST)

        for j in range(1, interp_per_gap + 1):
            alpha = j / (interp_per_gap + 1)
            try:
                interp = method.interpolate(m1, m2, alpha)
                out_path = output_dir / f"interp_{i:03d}_{j:02d}.png"
                cv2.imwrite(str(out_path), interp)
                interp_paths.append((i, j, out_path))
            except Exception as e:
                print(f"    Warning: interpolation failed at {i},{j}: {e}")

    # Chèn các frame nội suy vào đúng vị trí trong list
    for i, j, path in interp_paths:
        insert_idx = i + 1 + (j - 1)
        # Đảm bảo index không vượt quá độ dài
        if insert_idx < len(all_paths):
             all_paths.insert(insert_idx, path)
        else:
             all_paths.append(path)

    # Sort lại lần cuối để đảm bảo thứ tự
    all_paths = sorted(all_paths, key=lambda p: p.name)
    print(f"    Generated {len(interp_paths)} interpolated + {len(sorted_paths)} original = {len(all_paths)} total")
    return all_paths

print("Interpolation pipeline defined (Pearson only).")
print("Metrics updated: RMSE/MAE now calculate Water Level (Z) using Paper's Table 2.")

Interpolation pipeline defined (Pearson only).
Metrics updated: RMSE/MAE now calculate Water Level (Z) using Paper's Table 2.


In [13]:
import os
import pandas as pd
import yaml
import shutil
from ultralytics import YOLO
from pathlib import Path
from typing import Tuple

def train_evaluate_single_yolo(config: PipelineConfig, model_name: str) -> Tuple[pd.DataFrame, YOLO]:
    print("=" * 60)
    print(f"TRAINING SINGLE YOLO MODEL: {model_name}")
    print("=" * 60)

    # 1. Prepare Data structure for YOLO
    yolo_data_dir = config.processed_dir / "yolo_benchmark_data"
    for split in ["train", "val", "test"]:
        (yolo_data_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (yolo_data_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

    def copy_split(split_name, text_file_name):
        img_paths = load_split_file(config.split_dir / text_file_name)
        count = 0
        for img_path in img_paths:
            if img_path.exists():
                # Copy Image
                shutil.copy(str(img_path), str(yolo_data_dir / "images" / split_name / img_path.name))
                # Copy Label
                label_path = config.yolo_label_dir / (img_path.stem + ".txt")
                if label_path.exists():
                    shutil.copy(str(label_path), str(yolo_data_dir / "labels" / split_name / label_path.name))
                count += 1
        return count

    # Chỉ copy dữ liệu 1 lần nếu chưa có, hoặc copy lại để đảm bảo sạch
    n_train = copy_split("train", "train.txt")
    n_val = copy_split("val", "val.txt")
    n_test = copy_split("test", "test.txt")

    print(f"Dataset Prepared: {n_train} Train, {n_val} Val, {n_test} Test images")

    # 2. Create data.yaml
    data_yaml = {
        "path": str(yolo_data_dir.absolute()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {0: "water"}
    }
    yaml_path = yolo_data_dir / "data.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(data_yaml, f)

    results_data = []
    print(f"\n--- Processing {model_name} ---")
    run_name = model_name.replace(".pt", "")

    # Đường dẫn lưu weights
    save_dir = config.processed_dir / "yolo_benchmark" / run_name
    best_weight_path = save_dir / "weights" / "best.pt"

    # 3. Training
    # Kiểm tra xem đã train chưa để tránh train lại tốn thời gian
    if best_weight_path.exists():
        print(f"Found existing weights at {best_weight_path}. Loading...")
        try:
            model = YOLO(str(best_weight_path))
        except Exception as e:
            print(f"Error loading existing weights: {e}. Retraining...")
            model = YOLO(model_name)
            model.train(data=str(yaml_path), epochs=config.yolo_epochs, imgsz=config.yolo_image_size,
                        project=str(config.processed_dir / "yolo_benchmark"), name=run_name, exist_ok=True, verbose=False)
    else:
        print(f"Training {model_name} from scratch...")
        model = YOLO(model_name) # Tải pretrained model (ví dụ yolo11x-seg.pt)
        model.train(data=str(yaml_path),
                    epochs=config.yolo_epochs,
                    imgsz=config.yolo_image_size,
                    batch=8,
                    patience=30,
                    project=str(config.processed_dir / "yolo_benchmark"),
                    name=run_name,
                    exist_ok=True,
                    verbose=False)

    # 4. Evaluation
    print(f"Evaluating {model_name} on Test Set...")
    # Load best weights
    if best_weight_path.exists():
        model = YOLO(str(best_weight_path))

    metrics = model.val(split='test', verbose=False)

    # Extract metrics
    precision = metrics.seg.mp
    recall = metrics.seg.mr
    map50 = metrics.seg.map50
    map5095 = metrics.seg.map

    # Get model stats
    try:
        info = model.info(verbose=False)
        # info trả về (layers, params, gradients, flops)
        if isinstance(info, (tuple, list)):
            params_m = info[1] / 1e6
            gflops = info[3]
        else:
             params_m = 0
             gflops = 0
    except:
        params_m = 0
        gflops = 0

    size_mb = os.path.getsize(best_weight_path) / (1024 * 1024) if best_weight_path.exists() else 0
    inference_time = metrics.speed['inference']
    fps = 1000.0 / inference_time if inference_time > 0 else 0

    results_data.append({
        "Model": model_name,
        "P": round(precision, 3),
        "R": round(recall, 3),
        "mAP@0.5": round(map50, 3),
        "mAP@0.5:0.95": round(map5095, 3),
        "Params (M)": round(params_m, 2),
        "Size (MB)": round(size_mb, 1),
        "GFLOPs": round(gflops, 1),
        "FPS": round(fps, 1)
    })

    df = pd.DataFrame(results_data)
    return df, model

# === CẤU HÌNH MODEL CẦN CHẠY ===
target_model = "yolo11x-seg.pt"  # CHỈ CHẠY MODEL NÀY

result_df, trained_yolo_model = train_evaluate_single_yolo(cfg, target_model)

print("\n" + "="*80)
print("YOLO MODEL PERFORMANCE")
print("="*80)
print(result_df.to_markdown(index=False, numalign="center", stralign="center"))

# Lưu model vào dict để dùng cho pipeline phía sau
all_yolo_models = {target_model: trained_yolo_model}
benchmark_df = result_df

TRAINING SINGLE YOLO MODEL: yolo11x-seg.pt
Dataset Prepared: 42 Train, 10 Val, 19 Test images

--- Processing yolo11x-seg.pt ---
Found existing weights at /content/processed/yolo_benchmark/yolo11x-seg/weights/best.pt. Loading...
Evaluating yolo11x-seg.pt on Test Set...
Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11x-seg summary (fused): 204 layers, 62,003,283 parameters, 0 gradients, 295.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2162.6±552.5 MB/s, size: 246.8 KB)
val: Scanning /content/processed/yolo_benchmark_data/labels/test... 19 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 19/19 260.5it/s 0.1s
val: New cache created: /content/processed/yolo_benchmark_data/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.3it/s 1.5s
                   all         19         19      0.997          1      0.995   

In [14]:
def get_yolo_predicted_sequence(model: YOLO, mask_paths: List[Path], config: PipelineConfig) -> List[np.ndarray]:
    yolo_masks = []

    available_images = list(config.yolo_image_dir.glob("*.png"))
    timestamp_to_image = {}

    for img_path in available_images:
        ts = parse_timestamp(img_path)
        if ts is not None:
            timestamp_to_image[ts] = img_path

    print(f"Generating YOLO masks for {len(mask_paths)} images...")

    for mask_path in tqdm(mask_paths, desc="YOLO Inference"):
        img_path = None

        direct_path = config.yolo_image_dir / mask_path.name
        if direct_path.exists():
            img_path = direct_path
        else:
            ts = parse_timestamp(mask_path)
            if ts is not None and ts in timestamp_to_image:
                img_path = timestamp_to_image[ts]
            else:
                if ts is not None:
                    closest_ts = min(timestamp_to_image.keys(),
                                     key=lambda x: abs((x - ts).days),
                                     default=None)
                    if closest_ts is not None:
                        img_path = timestamp_to_image[closest_ts]

        if img_path is not None and img_path.exists():
            pred_mask = yolo_predict_mask(model, img_path, config)
        else:
            pred_mask = None

        if pred_mask is None:
            pred_mask = np.zeros(config.image_size, dtype=np.uint8)

        yolo_masks.append(pred_mask.astype(np.float32) / 255.0)

    return yolo_masks

In [15]:
from sklearn.model_selection import KFold
from torch.cuda.amp import autocast, GradScaler
from tqdm.auto import tqdm
import gc
import numpy as np

print("=" * 80)
print("EVALUATING YOLO MODEL IN K-FOLD CROSS-VALIDATION")
print("Pipeline: Satellite Image -> YOLO -> Mask -> ConvLSTM -> Prediction")
print("Metric: RMSE & MAE of Water Level (m) - Calculated via Interpolation")
print("=" * 80)

# Hàm đánh giá sử dụng mask_to_water_level (đã định nghĩa ở cell trước)
def evaluate_fold_scenarios_pipeline(conv_model, yolo_model, val_files, config):
    conv_model.eval()
    device = next(conv_model.parameters()).device

    gt_masks = [cv2.imread(str(p), cv2.IMREAD_GRAYSCALE).astype(np.float32)/255.0 for p in val_files]

    print(f"    Generating YOLO masks for {len(val_files)} images...")
    input_masks = get_yolo_predicted_sequence(yolo_model, val_files, config)

    results = {}

    for start_len in [3, 4, 5]:
        errors_mae = []
        errors_mse = []
        min_req = start_len + 1

        loop_range = range(len(input_masks) - min_req + 1)
        if len(loop_range) > 0:
            for i in tqdm(loop_range, desc=f"    Eval Scen (Input {start_len})", leave=False):
                with torch.no_grad():
                    # Chuẩn bị input sequence
                    seq_np = np.stack([m[None] for m in input_masks[i : i + start_len]])
                    input_tensor = torch.from_numpy(seq_np).float().unsqueeze(0).to(device)

                    # Dự báo frame tiếp theo
                    with autocast():
                        pred = conv_model(input_tensor).squeeze().cpu().float().numpy()

                    gt = gt_masks[i + start_len]

                    # --- SỬA ĐỔI QUAN TRỌNG: TÍNH TOÁN MỰC NƯỚC (MÉT) ---
                    # Gọi hàm mask_to_water_level đã định nghĩa ở Cell trên
                    # Input: Mask ảnh -> Output: Mực nước Z (m)
                    try:
                        pred_lvl = mask_to_water_level(pred)
                        gt_lvl = mask_to_water_level(gt)
                    except NameError:
                        raise NameError("Hàm 'mask_to_water_level' chưa được định nghĩa. Vui lòng chạy Cell định nghĩa hàm (chứa bảng tra AREA_POINTS) trước!")

                    # Tính lỗi tuyệt đối giữa các mức nước (đơn vị mét)
                    err = abs(pred_lvl - gt_lvl)
                    errors_mae.append(err)
                    errors_mse.append(err ** 2)

        if errors_mae:
            results[f"{start_len} images"] = {
                "MAE": np.mean(errors_mae),
                "RMSE": np.sqrt(np.mean(errors_mse))
            }
        else:
            results[f"{start_len} images"] = {"MAE": 0.0, "RMSE": 0.0}

    return results

# --- CẤU HÌNH K-FOLD ---
k_folds = 5
kf = KFold(n_splits=k_folds, shuffle=False)

# Load danh sách masks
all_mask_paths = load_split_file(cfg.split_dir / "train_masks.txt") + \
                 load_split_file(cfg.split_dir / "val_masks.txt") + \
                 load_split_file(cfg.split_dir / "test_masks.txt")
all_mask_paths = sorted(list(set(all_mask_paths)), key=lambda p: p.name)

model_comparison_results = {}
trained_convlstm_models = {}

# Kiểm tra xem dictionary model có tồn tại không, nếu không thì tạo dummy để test
if 'all_yolo_models' not in globals() or not all_yolo_models:
    print("⚠️ Warning: 'all_yolo_models' not found or empty. Using 'target_model' from previous steps.")
    try:
        # Thử load model từ biến target_model nếu có
        dummy_model = YOLO("yolo11x-seg.pt")
        all_yolo_models = {"yolo11x-seg.pt": dummy_model}
    except:
        print("❌ Error: Không tìm thấy model nào để chạy.")
        all_yolo_models = {}

# --- VÒNG LẶP CHÍNH ---
for yolo_name, yolo_model in all_yolo_models.items():
    print(f"\n{'='*80}")
    print(f"EVALUATING YOLO MODEL: {yolo_name}")
    print(f"{'='*80}")

    fold_results = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(all_mask_paths)):
        torch.cuda.empty_cache()
        gc.collect()

        print(f"\n--- Fold {fold + 1}/{k_folds} ---")

        fold_train_paths = [all_mask_paths[i] for i in train_idx]
        fold_val_paths = [all_mask_paths[i] for i in val_idx]

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Khởi tạo mô hình ConvLSTM (Forecasting)
        model = ReservoirConvLSTM(num_hidden=32, num_layers=3, shape=cfg.image_size).to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.convlstm_learning_rate)
        scaler = GradScaler()
        criterion = nn.MSELoss()

        dataset = TemporalMaskDataset(fold_train_paths, cfg.convlstm_sequence_length)
        loader = DataLoader(dataset, batch_size=cfg.convlstm_batch_size, shuffle=True, drop_last=True)

        model.train()
        k_epochs = 15 # Số epoch training mỗi fold

        print(f"    Training ConvLSTM ({len(fold_train_paths)} samples)...")
        for epoch in range(k_epochs):
            epoch_loss = 0
            pbar = tqdm(loader, desc=f"    Epoch {epoch+1}/{k_epochs}", leave=False)

            for seq, target in pbar:
                seq, target = seq.to(device), target.to(device)

                optimizer.zero_grad()

                with autocast():
                    pred = model(seq)
                    loss = criterion(pred, target)

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()

                epoch_loss += loss.item()
                pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        # Đánh giá Fold hiện tại
        print(f"    Evaluating Fold {fold+1}...")
        metrics = evaluate_fold_scenarios_pipeline(model, yolo_model, fold_val_paths, cfg)

        fold_record = {"Fold": f"Fold {fold + 1}"}
        for input_len, vals in metrics.items():
            fold_record[f"MAE_{input_len}"] = vals["MAE"]
            fold_record[f"RMSE_{input_len}"] = vals["RMSE"]

        fold_results.append(fold_record)
        print(f"    > Fold {fold+1} Results: {metrics}")

    # Lưu lại model đã train
    trained_convlstm_models[yolo_name] = model

    # Tổng hợp kết quả K-Fold
    df_kfold = pd.DataFrame(fold_results)
    avg_row = {"Fold": "Average"}
    for col in df_kfold.columns:
        if col != "Fold":
            avg_row[col] = df_kfold[col].mean()
    fold_results.append(avg_row)

    model_comparison_results[yolo_name] = pd.DataFrame(fold_results)

# --- HIỂN THỊ KẾT QUẢ TỔNG HỢP ---
print("\n" + "="*100)
print("K-FOLD RESULTS SUMMARY (Unit: Meters)")
print("="*100)

for yolo_name, df_result in model_comparison_results.items():
    print(f"\n{'='*80}")
    print(f"Results for: {yolo_name}")
    print("="*80)

    # Lọc và đổi tên cột để hiển thị đẹp
    cols_map = {
        "Fold": "Fold",
        "MAE_3 images": "MAE (3 img)", "MAE_4 images": "MAE (4 img)", "MAE_5 images": "MAE (5 img)",
        "RMSE_3 images": "RMSE (3 img)", "RMSE_4 images": "RMSE (4 img)", "RMSE_5 images": "RMSE (5 img)"
    }

    existing_cols = [c for c in cols_map.keys() if c in df_result.columns]
    df_display = df_result[existing_cols].rename(columns=cols_map)

    print(df_display.to_markdown(index=False, floatfmt=".4f", numalign="center"))

print("\n" + "="*100)
print("AVERAGE PERFORMANCE SUMMARY")
print("="*100)

summary_data = []
for yolo_name, df_result in model_comparison_results.items():
    avg_row = df_result[df_result["Fold"] == "Average"].iloc[0]
    summary_data.append({
        "YOLO Model": yolo_name,
        "MAE (3 img)": avg_row.get("MAE_3 images", 0),
        "MAE (4 img)": avg_row.get("MAE_4 images", 0),
        "MAE (5 img)": avg_row.get("MAE_5 images", 0),
        "RMSE (3 img)": avg_row.get("RMSE_3 images", 0),
        "RMSE (4 img)": avg_row.get("RMSE_4 images", 0),
        "RMSE (5 img)": avg_row.get("RMSE_5 images", 0),
    })

summary_df = pd.DataFrame(summary_data)print(summary_df.to_markdown(index=False, floatfmt=".4f", numalign="center"))

EVALUATING YOLO MODEL IN K-FOLD CROSS-VALIDATION
Pipeline: Satellite Image -> YOLO -> Mask -> ConvLSTM -> Prediction
Metric: RMSE & MAE of Water Level (m) - Calculated via Interpolation

EVALUATING YOLO MODEL: yolo11x-seg.pt

--- Fold 1/5 ---
    Training ConvLSTM (60 samples)...


/tmp/ipython-input-3690311300.py:115: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


    Epoch 1/15:   0%|          | 0/55 [00:00<?, ?it/s]

/tmp/ipython-input-3690311300.py:134: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


    Epoch 2/15:   0%|          | 0/55 [00:00<?, ?it/s]

...

YOLO Inference:   0%|          | 0/15 [00:00<?, ?it/s]

    Eval Scen (Input 3):   0%|          | 0/12 [00:00<?, ?it/s]

    Eval Scen (Input 4):   0%|          | 0/11 [00:00<?, ?it/s]

    Eval Scen (Input 5):   0%|          | 0/10 [00:00<?, ?it/s]

    > Fold 5 Results: {'3 images': {'MAE': np.float64(0.8108072916666677), 'RMSE': np.float64(0.8678283443959952)}, '4 images': {'MAE': np.float64(0.7364772727272721), 'RMSE': np.float64(0.7922225794386785)}, '5 images': {'MAE': np.float64(0.7985312500000077), 'RMSE': np.float64(0.8271669451597683)}}

K-FOLD RESULTS SUMMARY (Unit: Meters)

Results for: yolo11x-seg.pt
| Fold    |  MAE (3 img)  |  MAE (4 img)  |  MAE (5 img)  |  RMSE (3 img)  |  RMSE (4 img)  |  RMSE (5 img)  |
|:--------|:-------------:|:-------------:|:-------------:|:--------------:|:--------------:|:--------------:|
| Fold 1  |    2.1027     |    2.2439     |    1.9606     |     3.7093     |     3.8926     |     3.5470     |
| Fold 2  |    0.4634     |    0.4459     |    0.4408     |     0.5812     |     0.5764     |     0.5804     |
| Fold 3  |    0.8129     |    0.6746     |    0.5341     |     1.0942     |     0.9337     |     0.7086     |
| Fold 4  |    1.1265     |    1.0976     |    1.0670     |     1.1749     

In [16]:
from torch.cuda.amp import autocast
import numpy as np
import pandas as pd
import torch

print("\n" + "="*100)
print("WATER LEVEL FORECAST EVALUATION (RECURSIVE - METERS)")
print("Pipeline: Satellite -> YOLO -> Mask -> ConvLSTM -> Water Level (Z)")
print("Method: Interpolation using Table 2 from the paper")
print("="*100)

# ==========================================
# 1. DỮ LIỆU BẢNG TRA (PAPER TABLE 2)
# ==========================================
# Định nghĩa lại ở đây để đảm bảo cell chạy độc lập được
AREA_POINTS = np.array([
    0.00, 0.56, 0.73, 0.91, 1.12, 1.36, 1.68, 2.01, 2.18,
    2.36, 2.56, 2.78, 3.39, 4.02, 5.00, 6.13, 7.43, 9.00
])

WATER_LEVEL_POINTS = np.array([
    413.35, 420.00, 421.00, 422.00, 423.00, 424.00, 425.00, 426.00, 426.50,
    427.00, 427.50, 428.00, 429.00, 430.00, 431.00, 432.00, 433.00, 434.00
])

def get_water_level(mask: np.ndarray, config: PipelineConfig) -> float:
    """
    Chuyển đổi Mask -> Mực nước Z (m) thông qua nội suy.
    """
    if mask is None:
        return 0.0

    # Xử lý ngưỡng nhị phân cho mask
    threshold = 0.5 if mask.max() <= 1 else 127
    water_pixels = (mask > threshold).sum()

    # 1. Tính diện tích (km2)
    area_km2 = water_pixels * config.pixel_area_km2

    # 2. Nội suy ra mực nước (m)
    level_m = np.interp(area_km2, AREA_POINTS, WATER_LEVEL_POINTS)

    return level_m

# ==========================================
# 2. HÀM ĐÁNH GIÁ (RECURSIVE)
# ==========================================
def evaluate_water_levels_pipeline(conv_model, yolo_model, config,
                                   start_lengths=[3, 4, 5], forecast_horizon=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    conv_model.to(device)
    conv_model.eval()

    # Load dữ liệu test
    test_paths = load_split_file(config.split_dir / "test_masks.txt")
    test_paths = sorted(test_paths, key=lambda p: (parse_timestamp(p) or pd.Timestamp.max, p.name))

    # Load Ground Truth Masks
    gt_masks = [cv2.imread(str(p), cv2.IMREAD_GRAYSCALE).astype(np.float32)/255.0 for p in test_paths]

    print("Generating YOLO inputs for water level evaluation...")
    yolo_inputs = get_yolo_predicted_sequence(yolo_model, test_paths, config)

    results = []

    with torch.no_grad():
        for start_len in start_lengths:
            print(f"Processing Pipeline Input Length: {start_len} images...")

            # Lưu lỗi theo từng bước dự báo
            step_metrics = {s: {'errors': []} for s in range(1, forecast_horizon + 1)}
            min_req = start_len + forecast_horizon

            # Duyệt qua các chuỗi thời gian
            for i in range(len(yolo_inputs) - min_req + 1):
                # Chuỗi input ban đầu (Lấy từ YOLO)
                current_seq = [m.copy() for m in yolo_inputs[i : i + start_len]]

                # Vòng lặp dự báo đệ quy (Recursive Forecasting)
                for step in range(1, forecast_horizon + 1):
                    # Chuẩn bị input tensor
                    input_np = np.stack([m[None] for m in current_seq[-start_len:]])
                    input_tensor = torch.from_numpy(input_np).float().unsqueeze(0).to(device)

                    # Dự báo frame tiếp theo
                    with autocast():
                        pred = conv_model(input_tensor).squeeze().cpu().float().numpy()

                    # Lấy Ground Truth tương ứng
                    gt_idx = i + start_len + step - 1

                    # --- TÍNH TOÁN MỰC NƯỚC (QUAN TRỌNG) ---
                    level_gt = get_water_level(gt_masks[gt_idx], config) # Z thực tế (m)
                    level_pred = get_water_level(pred, config)           # Z dự báo (m)
                    # ---------------------------------------

                    # Lưu lỗi tuyệt đối (m)
                    step_metrics[step]['errors'].append(abs(level_gt - level_pred))

                    # Cập nhật chuỗi input cho bước tiếp theo
                    current_seq.append(pred)

            # Tổng hợp kết quả
            for step in range(1, forecast_horizon + 1):
                errs = np.array(step_metrics[step]['errors'])
                if len(errs) > 0:
                    results.append({
                        "Input Images": f"{start_len} images",
                        "Forecast Step": f"Step {step}",
                        "MAE (m)": np.mean(errs),            # Mean Absolute Error
                        "RMSE (m)": np.sqrt(np.mean(errs ** 2)) # Root Mean Square Error
                    })

    df = pd.DataFrame(results)
    return df

# ==========================================
# 3. CHẠY ĐÁNH GIÁ
# ==========================================

# Kiểm tra model
if 'all_yolo_models' in globals() and all_yolo_models:
    yolo_name = list(all_yolo_models.keys())[0]
    yolo_model = all_yolo_models[yolo_name]
else:
    print("Warning: Không tìm thấy model YOLO, sử dụng dummy 'yolo11x-seg.pt'")
    yolo_name = "yolo11x-seg.pt"
    yolo_model = YOLO(yolo_name)

print(f"\nEvaluating Water Levels with: {yolo_name}")

if 'trained_convlstm_models' in globals() and yolo_name in trained_convlstm_models:
    convlstm_model = trained_convlstm_models[yolo_name]
else:
    print(f"Training ConvLSTM for {yolo_name} (Fallback)...")
    # Nếu chưa train thì khởi tạo mới (lưu ý: kết quả sẽ tệ nếu chưa train)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    convlstm_model = ReservoirConvLSTM(num_hidden=32, num_layers=3, shape=cfg.image_size).to(device)

wl_results = evaluate_water_levels_pipeline(convlstm_model, yolo_model, cfg)

print(f"\nWater Level Results for {yolo_name}:")print(wl_results.to_markdown(index=False, floatfmt=".4f", numalign="center"))


WATER LEVEL FORECAST EVALUATION (RECURSIVE - METERS)
Pipeline: Satellite -> YOLO -> Mask -> ConvLSTM -> Water Level (Z)
Method: Interpolation using Table 2 from the paper

Evaluating Water Levels with: yolo11x-seg.pt
Generating YOLO inputs for water level evaluation...
Generating YOLO masks for 13 images...


YOLO Inference:   0%|          | 0/13 [00:00<?, ?it/s]

Processing Pipeline Input Length: 3 images...


/tmp/ipython-input-784329814.py:86: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Processing Pipeline Input Length: 4 images...
Processing Pipeline Input Length: 5 images...

Water Level Results for yolo11x-seg.pt:
| Input Images   | Forecast Step   |  MAE (m)  |  RMSE (m)  |
|:---------------|:----------------|:---------:|:----------:|
| 3 images       | Step 1          |  0.8745   |   0.9098   |
| 3 images       | Step 2          |  0.8254   |   0.8471   |
| 3 images       | Step 3          |  0.7407   |   0.7638   |
| 4 images       | Step 1          |  0.8789   |   0.9034   |
| 4 images       | Step 2          |  0.7840   |   0.8085   |
| 4 images       | Step 3          |  0.6829   |   0.7009   |
| 5 images       | Step 1          |  0.8579   |   0.8851   |
| 5 images       | Step 2          |  0.7527   |   0.7702   |
| 5 images       | Step 3          |  0.7314   |   0.7507   |


In [17]:
from torch.cuda.amp import autocast
import numpy as np
import pandas as pd
import torch

print("=" * 80)
print("RECURSIVE FORECASTING EVALUATION (MULTI-STEP)")
print("Pipeline: Satellite -> YOLO -> Mask -> ConvLSTM (Recursive) -> Water Level (Z)")
print("Metrics included: Water Level (m) AND Pixel-Level Error")
print("=" * 80)

def evaluate_recursive_steps_pipeline(conv_model, yolo_model, config,
                                      start_lengths=[3, 4, 5], forecast_horizon=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    conv_model.to(device)
    conv_model.eval()

    # Load test masks
    test_masks_paths = load_split_file(config.split_dir / "test_masks.txt")
    test_masks_paths = sorted(test_masks_paths, key=lambda p: (parse_timestamp(p) or pd.Timestamp.max, p.name))

    # Load Ground Truth Masks
    gt_masks = [cv2.imread(str(p), cv2.IMREAD_GRAYSCALE).astype(np.float32)/255.0 for p in test_masks_paths]

    print("Generating YOLO inputs for test set...")
    yolo_inputs = get_yolo_predicted_sequence(yolo_model, test_masks_paths, config)

    detailed_results = []

    with torch.no_grad():
        for start_len in start_lengths:
            print(f"Processing Pipeline Input Length: {start_len} images...")

            # Khởi tạo dictionary chứa các metrics
            step_metrics = {
                s: {
                    'wl_sq_errors': [], 'wl_abs_errors': [], # Water Level metrics (m)
                    'pix_sq_errors': [], 'pix_abs_errors': [], # Pixel metrics (0-255)
                    'ious': []
                } for s in range(1, forecast_horizon + 1)
            }
            min_required = start_len + forecast_horizon

            for i in range(len(yolo_inputs) - min_required + 1):
                current_seq = [m.copy() for m in yolo_inputs[i : i + start_len]]

                for step in range(1, forecast_horizon + 1):
                    # 1. Dự báo
                    input_np = np.stack([m[None] for m in current_seq[-start_len:]])
                    input_tensor = torch.from_numpy(input_np).float().unsqueeze(0).to(device)

                    with autocast():
                         pred_tensor = conv_model(input_tensor)
                    pred_np = pred_tensor.squeeze().cpu().float().numpy() # Range 0-1

                    # 2. Lấy Ground Truth
                    gt_idx = i + start_len + step - 1
                    gt_np = gt_masks[gt_idx]

                    # --- A. TÍNH WATER LEVEL ERROR (MÉT) ---
                    try:
                        pred_lvl = mask_to_water_level(pred_np)
                        gt_lvl = mask_to_water_level(gt_np)
                        wl_err = abs(pred_lvl - gt_lvl)
                    except NameError:
                         wl_err = 0.0 # Fallback nếu chưa chạy cell định nghĩa hàm

                    # --- B. TÍNH PIXEL-LEVEL ERROR (HÌNH DẠNG) ---
                    # Chuyển về 0-255 để tính lỗi pixel chuẩn hơn
                    pred_255 = pred_np * 255
                    gt_255 = gt_np * 255

                    # Pixel Absolute Error (MAE Pixel)
                    pix_abs = np.mean(np.abs(pred_255 - gt_255))
                    # Pixel Squared Error (RMSE Pixel)
                    pix_sq = np.mean((pred_255 - gt_255) ** 2)

                    # --- C. TÍNH IOU ---
                    pred_bin = (pred_np > 0.5).astype(np.uint8)
                    gt_bin = (gt_np > 0.5).astype(np.uint8) if gt_np.max() <= 1 else (gt_np > 127).astype(np.uint8)
                    iou = compute_iou(gt_bin, pred_bin)

                    # --- LƯU TRỮ ---
                    step_metrics[step]['wl_sq_errors'].append(wl_err ** 2)
                    step_metrics[step]['wl_abs_errors'].append(wl_err)

                    step_metrics[step]['pix_sq_errors'].append(pix_sq)
                    step_metrics[step]['pix_abs_errors'].append(pix_abs)

                    step_metrics[step]['ious'].append(iou)

                    # Cập nhật sequence
                    current_seq.append(pred_np)

            # Tổng hợp kết quả
            for step in range(1, forecast_horizon + 1):
                m = step_metrics[step]
                if m['wl_abs_errors']:
                    detailed_results.append({
                        "Input Images": f"{start_len} images",
                        "Forecast Step": f"Step {step}",
                        # Water Level Metrics
                        "WL_MAE (m)": np.mean(m['wl_abs_errors']),
                        "WL_RMSE (m)": np.sqrt(np.mean(m['wl_sq_errors'])),
                        # Pixel Metrics
                        "Pixel_MAE": np.mean(m['pix_abs_errors']),
                        "Pixel_RMSE": np.sqrt(np.mean(m['pix_sq_errors'])),
                        # Shape Metric
                        "IoU": np.mean(m['ious'])
                    })

    df = pd.DataFrame(detailed_results)
    return df

# --- PHẦN CHẠY ---
if 'all_yolo_models' in globals() and all_yolo_models:
    yolo_name = list(all_yolo_models.keys())[0]
    yolo_model = all_yolo_models[yolo_name]
else:
    print("Warning: Sử dụng dummy model YOLO.")
    yolo_name = "yolo11x-seg.pt"
    yolo_model = YOLO(yolo_name)

if 'trained_convlstm_models' in globals() and yolo_name in trained_convlstm_models:
    convlstm_model = trained_convlstm_models[yolo_name]
else:
    print(f"Khởi tạo ConvLSTM mới cho {yolo_name}...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    convlstm_model = ReservoirConvLSTM(num_hidden=32, num_layers=3, shape=cfg.image_size).to(device)

print(f"\nEvaluating Recursive Forecasting for: {yolo_name}")
step_results = evaluate_recursive_steps_pipeline(convlstm_model, yolo_model, cfg)

print(f"\nResults for {yolo_name}:")
print(step_results.to_markdown(index=False, floatfmt=".4f", numalign="center"))

RECURSIVE FORECASTING EVALUATION (MULTI-STEP)
Pipeline: Satellite -> YOLO -> Mask -> ConvLSTM (Recursive) -> Water Level (Z)
Metrics included: Water Level (m) AND Pixel-Level Error

Evaluating Recursive Forecasting for: yolo11x-seg.pt
Generating YOLO inputs for test set...
Generating YOLO masks for 13 images...


YOLO Inference:   0%|          | 0/13 [00:00<?, ?it/s]

Processing Pipeline Input Length: 3 images...


/tmp/ipython-input-1990355661.py:52: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Processing Pipeline Input Length: 4 images...
Processing Pipeline Input Length: 5 images...

Results for yolo11x-seg.pt:
| Input Images   | Forecast Step   |  WL_MAE (m)  |  WL_RMSE (m)  |  Pixel_MAE  |  Pixel_RMSE  |  IoU   |
|:---------------|:----------------|:------------:|:-------------:|:-----------:|:------------:|:------:|
| 3 images       | Step 1          |    0.8745    |    0.9098     |   10.4958   |   38.5317    | 0.7752 |
| 3 images       | Step 2          |    0.8254    |    0.8471     |   9.8778    |   37.3040    | 0.7875 |
| 3 images       | Step 3          |    0.7407    |    0.7638     |   9.4718    |   36.7533    | 0.8001 |
| 4 images       | Step 1          |    0.8789    |    0.9034     |   10.2494   |   37.9038    | 0.7803 |
| 4 images       | Step 2          |    0.7840    |    0.8085     |   9.7502    |   37.0490    | 0.7907 |
| 4 images       | Step 3          |    0.6829    |    0.7009     |   9.0348    |   35.3930    | 0.8098 |
| 5 images       | Step 1      

In [18]:
print("="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)

print("\n1. YOLO BENCHMARK (Segmentation Performance):")
print(benchmark_df[["Model", "mAP@0.5", "mAP@0.5:0.95", "FPS"]].to_string(index=False))

print("\n2. K-FOLD AVERAGE (Water Level Prediction Error):")
print(summary_df.to_string(index=False))

print("\n3. BEST PERFORMING MODEL BY METRIC:")
best_map = benchmark_df.loc[benchmark_df["mAP@0.5:0.95"].idxmax(), "Model"]
best_mae_3 = summary_df.loc[summary_df["MAE (3 img)"].idxmin(), "YOLO Model"]
best_mae_5 = summary_df.loc[summary_df["MAE (5 img)"].idxmin(), "YOLO Model"]

print(f"  - Best mAP@0.5:0.95: {best_map}")
print(f"  - Best MAE (3 images): {best_mae_3}")
print(f"  - Best MAE (5 images): {best_mae_5}")

FINAL RESULTS SUMMARY

1. YOLO BENCHMARK (Segmentation Performance):
         Model  mAP@0.5  mAP@0.5:0.95  FPS
yolo11x-seg.pt    0.995         0.676 22.6

2. K-FOLD AVERAGE (Water Level Prediction Error):
    YOLO Model  MAE (3 img)  MAE (4 img)  MAE (5 img)  RMSE (3 img)  RMSE (4 img)  RMSE (5 img)
yolo11x-seg.pt     1.063263     1.039698     0.960182      1.485481      1.468068      1.355754

3. BEST PERFORMING MODEL BY METRIC:
  - Best mAP@0.5:0.95: yolo11x-seg.pt
  - Best MAE (3 images): yolo11x-seg.pt
  - Best MAE (5 images): yolo11x-seg.pt


In [19]:
import torch
from ultralytics import YOLO

def count_parameters(model):
    """Count trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 40)
print("PIPELINE PARAMETER COUNTS")
print("=" * 40)

try:
    convlstm = ReservoirConvLSTM()
    convlstm_params = count_parameters(convlstm)
    print(f"ReservoirConvLSTM (Forecasting): {convlstm_params:,} parameters")
except NameError:
    print("ReservoirConvLSTM class not defined. Please run the model definition cell first.")
except Exception as e:
    print(f"Error counting ConvLSTM parameters: {e}")

PIPELINE PARAMETER COUNTS
ReservoirConvLSTM (Forecasting): 19,808,993 parameters
